# Advertising Benchmark

This notebook imports saved advertising runs from `results/advertising`. It is analysis-only and does not run training.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from mfc.visualization import (
    best_runs_by_label,
    discrete_transport_tv_bound_table,
    load_runs,
    objective_table,
    plot_advertising_diagnostics,
    plot_state_flow,
    plot_validation_rewards,
    runtime_table,
)

ENV = 'advertising'
RESULTS_ROOT = ROOT / 'results'
runs = load_runs(RESULTS_ROOT, env=ENV)
print(f'Loaded {len(runs)} saved runs from {RESULTS_ROOT / ENV}')

## Validation Reward

Mean validation reward over training, with one standard deviation across seeds. The reward balances customer acquisition against advertising cost.

In [ ]:
if not runs:
    print('No saved runs yet. Run scripts/run.py before executing the analysis cells.')
else:
    fig, ax = plt.subplots(figsize=(8, 4.5))
    plot_validation_rewards(runs, env=ENV, horizon=5, flow='exact', ax=ax)
    ax.set_title('Advertising validation reward')
    plt.show()

## Customer Share and Ad Intensity

For each selected learned policy, this plot shows the customer share and the average probability of displaying an ad over the validation horizon.

In [ ]:
for run in best_runs_by_label(runs):
    fig, ax = plt.subplots(figsize=(8, 4.5))
    plot_advertising_diagnostics(run, ax=ax)
    meta = run['metadata']
    ax.set_title(f"{meta['algorithm']}, perturbation={meta['perturbation']}")
    plt.show()

## State Distribution Flow

State-probability trajectories for not-customers and customers under the learned policies.

In [ ]:
for run in best_runs_by_label(runs):
    fig, ax = plt.subplots(figsize=(8, 4.5))
    plot_state_flow(run, ax=ax)
    meta = run['metadata']
    ax.set_title(f"State flow: {meta['algorithm']}, perturbation={meta['perturbation']}")
    plt.show()

## Transport Perturbation Bound

For simplex transport, the total-variation distance between the perturbed and unperturbed population laws is bounded by the selected `lambda`.

In [ ]:
tv_bounds = discrete_transport_tv_bound_table(runs)
display(tv_bounds if not tv_bounds.empty else pd.DataFrame({'message': ['No transport runs found yet.']}))

## Objective and Runtime Tables

Final validation reward, estimated simulator budget, and runtime summaries.

In [ ]:
display(objective_table(runs))
display(runtime_table(runs))